# RSI Backtest Report

**Scope:** 21 FX pairs · RSI offset 17 · costs excluded.

Results use closed trades exported by TradingView. Open positions are retained in the merged CSV and excluded from realized performance. This report assesses the supplied rules; it does not independently rerun the strategy on candle data.

## Data preparation

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import hashlib
import itertools
input_folder = Path('C:\\Users\\akbar\\OneDrive\\Desktop\\data\\xlsxs')
output_folder = Path('C:\\Users\\akbar\\OneDrive\\Desktop\\data\\final')
output_folder.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 220)
files = sorted(input_folder.glob('*.xlsx'))
assert len(files) == 21, 'تعداد فایل\u200cها باید ۲۱ باشه.'
pass
all_trades = []
file_checks = []
settings = []
trade_hashes = set()
for file in files:
    sheets = pd.read_excel(file, sheet_name=None)
    required = {'Properties', 'Trades', 'Performance', 'Trades analysis', 'Risk-adjusted performance'}
    assert required.issubset(sheets), f'شیت ناقص: {file.name}'
    properties_sheet = pd.read_excel(file, sheet_name='Properties', keep_default_na=False)
    props = dict(properties_sheet.iloc[:, :2].itertuples(index=False, name=None))
    symbol = props['Symbol']
    raw = sheets['Trades']
    digest = hashlib.sha256(raw.iloc[:, :].drop(columns=[raw.columns[4]]).to_csv(index=False).encode()).hexdigest()
    assert digest not in trade_hashes, f'محتوای تکراری: {file.name}'
    trade_hashes.add(digest)
    assert raw['Trade number'].notna().all()
    entries = raw[raw['Type'].str.startswith('Entry', na=False)].copy()
    exits = raw[raw['Type'].str.startswith('Exit', na=False)].copy()
    assert len(entries) + len(exits) == len(raw), f'نوع ردیف ناشناخته: {symbol}'
    exits = exits[exits['Date and time'].astype(str).ne('Open')].copy()
    assert not entries['Trade number'].duplicated().any()
    assert not exits['Trade number'].duplicated().any()
    assert set(exits['Trade number']).issubset(set(entries['Trade number']))
    price_column = raw.columns[4]
    joined = entries.merge(exits, on='Trade number', how='left', suffixes=('_entry', '_exit'), validate='one_to_one')
    table = pd.DataFrame({'symbol': symbol, 'trade_number': joined['Trade number'].astype(int), 'direction': joined['Type_entry'].str.replace('Entry ', '', regex=False), 'entry_time': pd.to_datetime(joined['Date and time_entry'], errors='raise'), 'exit_time': pd.to_datetime(joined['Date and time_exit'], errors='raise'), 'entry_price': joined[price_column + '_entry'], 'exit_price': joined[price_column + '_exit'], 'quantity': joined['Size (qty)_entry'], 'entry_value_quote': joined['Size (value)_entry'], 'pnl_usd': joined['Net PnL USD_exit'], 'trade_return_pct': joined['Return %_exit'], 'commission_usd': joined['Commission USD_exit'], 'mfe_usd': joined['Favorable excursion USD_exit'], 'mae_usd': joined['Adverse excursion USD_exit'], 'duration_bars': joined['Duration (bars)_exit'], 'tv_cumulative_pnl_usd': joined['Cumulative PnL USD_exit'], 'exit_signal': joined['Signal_exit'], 'quote_currency': props['Currency'], 'source_file': file.name})
    table['status'] = np.where(table['exit_time'].notna(), 'closed', 'open')
    table['trade_id'] = table['symbol'] + ':' + table['trade_number'].astype(str)
    table = table.sort_values('entry_time').reset_index(drop=True)
    closed = table[table['status'] == 'closed']
    assert table[['entry_time', 'entry_price', 'quantity', 'direction']].notna().all().all()
    assert closed[['exit_price', 'pnl_usd', 'duration_bars']].notna().all().all()
    assert (closed['exit_time'] >= closed['entry_time']).all()
    assert (table['entry_time'].iloc[1:].reset_index(drop=True) >= table['exit_time'].iloc[:-1].reset_index(drop=True)).all(), f'هم\u200cپوشانی یا معاملهٔ باز میان تاریخچه: {symbol}'
    direction_sign = np.where(closed['direction'] == 'long', 1, -1)
    price_pnl = (closed['exit_price'] - closed['entry_price']) * direction_sign
    assert (np.sign(price_pnl) == np.sign(closed['pnl_usd'])).all()
    analysis = sheets['Trades analysis'].set_index(sheets['Trades analysis'].columns[0])
    performance = sheets['Performance'].set_index(sheets['Performance'].columns[0])
    reported_count = int(analysis.loc['Total trades'].iloc[0])
    reported_open = int(analysis.loc['Total open trades'].iloc[0])
    reported_pnl = float(performance.loc['Net profit'].iloc[0])
    assert len(closed) == reported_count
    assert (table['status'] == 'open').sum() == reported_open
    assert (closed['pnl_usd'] > 0).sum() == int(analysis.loc['Total winners'].iloc[0])
    assert (closed['pnl_usd'] < 0).sum() == int(analysis.loc['Total losers'].iloc[0])
    pnl_difference = closed['pnl_usd'].sum() - reported_pnl
    assert abs(pnl_difference) <= 0.005 * len(closed) + 0.011, f'اختلاف فراتر از حد گردشدن: {symbol}'
    start_text, end_text = props['Backtesting range'].split('—')
    start = pd.to_datetime(start_text.strip())
    end = pd.to_datetime(end_text.strip())
    file_checks.append({'symbol': symbol, 'closed_trades': len(closed), 'open_trades': reported_open, 'coverage_start': start, 'coverage_end': end, 'first_entry': table['entry_time'].min(), 'reported_pnl_usd': reported_pnl, 'sum_trade_pnl_usd': closed['pnl_usd'].sum(), 'rounding_difference_usd': pnl_difference, 'max_days_between_entries': table['entry_time'].diff().dt.total_seconds().max() / 86400})
    settings.append({'symbol': symbol, **props})
    all_trades.append(table)
trades = pd.concat(all_trades, ignore_index=True)
checks = pd.DataFrame(file_checks)
settings = pd.DataFrame(settings)
expected_currencies = ['AUD', 'CAD', 'CHF', 'EUR', 'GBP', 'NZD', 'USD']
expected_pairs = {frozenset(pair) for pair in itertools.combinations(expected_currencies, 2)}
actual_pairs = {frozenset([s.split(':')[1][:3], s.split(':')[1][3:]]) for s in trades['symbol'].unique()}
assert actual_pairs == expected_pairs
assert checks['symbol'].nunique() == 21
assert not trades['trade_id'].duplicated().any()
for column, expected in {'Timeframe': '1 hour', 'Chart type': 'Candles', 'RSI length (close)': '14', 'Long entry: cross below': '30', 'Short entry: cross above': '70', 'RSI target distance from signal RSI': '17', 'Fixed position quantity (symbol units)': '1000'}.items():
    assert settings[column].astype(str).eq(expected).all(), column
for column in settings.columns:
    if column not in ['symbol', 'Symbol', 'Trading range', 'Backtesting range', 'Currency', 'Tick size']:
        assert settings[column].astype(str).nunique() == 1, f'تنظیم متفاوت: {column}'
trades.to_csv(output_folder / 'merged_trades.csv', index=False, encoding='utf-8-sig')
checks.to_csv(output_folder / 'data_checks.csv', index=False, encoding='utf-8-sig')
settings.to_csv(output_folder / 'source_settings.csv', index=False, encoding='utf-8-sig')
pass
pass

trades = pd.read_csv(output_folder / 'merged_trades.csv', parse_dates=['entry_time', 'exit_time'])
closed = trades[trades['status'] == 'closed'].copy()

def profit_factor(pnl):
    profit = pnl[pnl > 0].sum()
    loss = -pnl[pnl < 0].sum()
    if loss > 0:
        return profit / loss
    return np.inf if profit > 0 else np.nan
pair_rows = []
for symbol, group in closed.groupby('symbol'):
    long = group[group['direction'] == 'long']['pnl_usd']
    short = group[group['direction'] == 'short']['pnl_usd']
    smaller_count = min(len(long), len(short))
    count_ratio = max(len(long), len(short)) / smaller_count if smaller_count else np.inf
    pf_long = profit_factor(long)
    pf_short = profit_factor(short)
    valid_pf = np.isfinite(pf_long) and np.isfinite(pf_short) and (min(pf_long, pf_short) > 0)
    pf_ratio = max(pf_long, pf_short) / min(pf_long, pf_short) if valid_pf else np.nan
    pair_rows.append({'symbol': symbol, 'trades': len(group), 'long_count': len(long), 'short_count': len(short), 'net_pnl_usd': group['pnl_usd'].sum(), 'long_pnl_usd': long.sum(), 'short_pnl_usd': short.sum(), 'pf': profit_factor(group['pnl_usd']), 'pf_long': pf_long, 'pf_short': pf_short, 'count_ratio': count_ratio, 'pf_ratio': pf_ratio, 'at_least_100': len(group) >= 100, 'at_least_800': len(group) >= 800, 'count_balance_ok': count_ratio <= 1.2 + 1e-12, 'pf_balance_ok': bool(valid_pf and pf_ratio <= 1.1 + 1e-12)})
pair_report = pd.DataFrame(pair_rows)
pair_report['rule_1_ok'] = pair_report[['at_least_800', 'pf_balance_ok']].all(axis=1)
pair_report['full_pair_rules_ok'] = pair_report[['rule_1_ok', 'count_balance_ok']].all(axis=1)
pair_report.to_csv(output_folder / 'pair_rules.csv', index=False, encoding='utf-8-sig')
pass
winners = pair_report[pair_report['net_pnl_usd'] > 0]
losers = pair_report[pair_report['net_pnl_usd'] < 0]
flat_count = int((pair_report['net_pnl_usd'] == 0).sum())
mean_profit = winners['net_pnl_usd'].mean()
mean_loss = losers['net_pnl_usd'].abs().mean()
mean_comparison = mean_profit >= mean_loss if len(winners) and len(losers) else None
overall = pd.DataFrame([{'rule': 'حداقل ۱۰۰ معامله برای تک\u200cتک ۲۱ جفت', 'value': int(pair_report['trades'].min()), 'passed': bool(pair_report['at_least_100'].all())}, {'rule': 'حداقل ۱۴ جفت سودده', 'value': len(winners), 'passed': len(winners) >= 14}, {'rule': 'میانگین سود جفت\u200cهای مثبت ≥ میانگین قدرمطلق زیان جفت\u200cهای منفی', 'value': mean_profit / mean_loss if len(losers) and mean_loss else np.nan, 'passed': mean_comparison}])
overall.to_csv(output_folder / 'overall_rules.csv', index=False, encoding='utf-8-sig')
pass
pass
pass
pass
pass
pass
pass
pair_report['long_pct'] = pair_report['long_count'] / pair_report['trades'] * 100
pair_report['short_pct'] = pair_report['short_count'] / pair_report['trades'] * 100
pair_report['rule_1_ok'] = pair_report[['at_least_800', 'count_balance_ok', 'pf_balance_ok']].all(axis=1)
pair_report.to_csv(output_folder / 'pair_rules.csv', index=False, encoding='utf-8-sig')

def show_table(table):
    print(table.round(4).to_string(index=False))

print(f"Pairs: {trades['symbol'].nunique()} | Closed trades: {len(closed):,} | Open trades: {(trades['status'] == 'open').sum()}")
print('Validation: unique trade IDs, matched exits, chronological order, trade counts and rounding reconciliation checked.')
print('Independent merged dataset: merged_trades.csv')


Pairs: 21 | Closed trades: 49,520 | Open trades: 4
Validation: unique trade IDs, matched exits, chronological order, trade counts and rounding reconciliation checked.
Independent merged dataset: merged_trades.csv


## 1. Minimum sample size

Requirement: approximately 800 or more closed trades per pair. Counts and directional shares are shown below.

In [2]:
show_table(pair_report[['symbol', 'trades', 'long_count', 'short_count', 'long_pct', 'short_pct', 'at_least_800']])
print(f"Result: {pair_report['at_least_800'].sum()}/21 pairs meet the minimum.")

   symbol  trades  long_count  short_count  long_pct  short_pct  at_least_800
FX:AUDCAD    2150        1058         1092   49.2093    50.7907          True
FX:AUDCHF    2329        1210         1119   51.9536    48.0464          True
FX:AUDNZD    2051        1046         1005   50.9995    49.0005          True
FX:AUDUSD    2461        1285         1176   52.2145    47.7855          True
FX:CADCHF    2383        1200         1183   50.3567    49.6433          True
FX:EURAUD    2441        1224         1217   50.1434    49.8566          True
FX:EURCAD    2466        1221         1245   49.5134    50.4866          True
FX:EURCHF    2071        1082          989   52.2453    47.7547          True
FX:EURGBP    2569        1298         1271   50.5255    49.4745          True
FX:EURNZD    2297        1152         1145   50.1524    49.8476          True
FX:EURUSD    2568        1320         1248   51.4019    48.5981          True
FX:GBPAUD    2323        1127         1196   48.5149    51.4851 

## 2. Long/short count balance

Requirement: the larger directional count must not exceed the smaller by more than 20%. Ratio limit: 1.20.

In [3]:
show_table(pair_report[['symbol', 'long_count', 'short_count', 'count_ratio', 'count_balance_ok']])
print(f"Result: {pair_report['count_balance_ok'].sum()}/21 pairs pass.")

   symbol  long_count  short_count  count_ratio  count_balance_ok
FX:AUDCAD        1058         1092       1.0321              True
FX:AUDCHF        1210         1119       1.0813              True
FX:AUDNZD        1046         1005       1.0408              True
FX:AUDUSD        1285         1176       1.0927              True
FX:CADCHF        1200         1183       1.0144              True
FX:EURAUD        1224         1217       1.0058              True
FX:EURCAD        1221         1245       1.0197              True
FX:EURCHF        1082          989       1.0940              True
FX:EURGBP        1298         1271       1.0212              True
FX:EURNZD        1152         1145       1.0061              True
FX:EURUSD        1320         1248       1.0577              True
FX:GBPAUD        1127         1196       1.0612              True
FX:GBPCAD        1203         1207       1.0033              True
FX:GBPCHF        1243         1166       1.0660              True
FX:GBPNZD 

## 3. Long/short profit-factor balance

Requirement: the larger directional PF must not exceed the smaller by more than 10%. Ratio limit: 1.10. Positive net profit in both directions is not imposed as an additional condition.

In [4]:
show_table(pair_report[['symbol', 'pf_long', 'pf_short', 'pf_ratio', 'pf_balance_ok']])
print(f"Result: {pair_report['pf_balance_ok'].sum()}/21 pairs pass.")

   symbol  pf_long  pf_short  pf_ratio  pf_balance_ok
FX:AUDCAD   0.9979    1.3190    1.3217          False
FX:AUDCHF   0.9936    1.0472    1.0540           True
FX:AUDNZD   0.9779    1.2014    1.2285          False
FX:AUDUSD   1.0161    1.1091    1.0915           True
FX:CADCHF   0.9465    1.0610    1.1210          False
FX:EURAUD   0.9825    1.0022    1.0200           True
FX:EURCAD   1.0163    1.0816    1.0643           True
FX:EURCHF   1.1380    1.2627    1.1096          False
FX:EURGBP   1.0609    0.9073    1.1693          False
FX:EURNZD   1.1185    0.9431    1.1860          False
FX:EURUSD   0.9135    1.0617    1.1622          False
FX:GBPAUD   1.0579    1.0270    1.0300           True
FX:GBPCAD   1.0967    1.1548    1.0530           True
FX:GBPCHF   1.0014    1.0540    1.0525           True
FX:GBPNZD   1.1071    1.0532    1.0512           True
FX:GBPUSD   0.9238    1.0264    1.1111          False
FX:NZDCAD   1.1578    1.1468    1.0096           True
FX:NZDCHF   1.0472    1.2354

## 4. Preliminary sample requirement

Requirement: at least 100 closed trades on every pair; a total of 2,100 alone is insufficient.

In [5]:
print('Result:', 'PASS' if pair_report['at_least_100'].all() else 'FAIL')
print(f"Minimum per pair: {pair_report['trades'].min():,}. Total closed trades: {len(closed):,}.")

Result: PASS
Minimum per pair: 2,051. Total closed trades: 49,520.


## 5. Number of profitable pairs

Requirement: at least 14 of 21 pairs must have positive net realized profit.

In [6]:
profitability = pair_report[['symbol', 'net_pnl_usd']].copy()
profitability['status'] = np.select([profitability['net_pnl_usd'] > 0, profitability['net_pnl_usd'] < 0], ['Profitable', 'Losing'], default='Breakeven')
show_table(profitability)
print(f"Result: {'PASS' if len(winners) >= 14 else 'FAIL'} — {len(winners)}/21 profitable pairs.")

   symbol  net_pnl_usd     status
FX:AUDCAD       386.67 Profitable
FX:AUDCHF        70.45 Profitable
FX:AUDNZD       174.46 Profitable
FX:AUDUSD       242.62 Profitable
FX:CADCHF         3.13 Profitable
FX:EURAUD       -39.66     Losing
FX:EURCAD       230.28 Profitable
FX:EURCHF       517.75 Profitable
FX:EURGBP       -95.99     Losing
FX:EURNZD       129.81 Profitable
FX:EURUSD       -88.52     Losing
FX:GBPAUD       262.83 Profitable
FX:GBPCAD       682.49 Profitable
FX:GBPCHF       154.70 Profitable
FX:GBPNZD       500.46 Profitable
FX:GBPUSD      -170.47     Losing
FX:NZDCAD       402.25 Profitable
FX:NZDCHF       425.17 Profitable
FX:NZDUSD      -288.35     Losing
FX:USDCAD       -82.56     Losing
FX:USDCHF      -103.42     Losing
Result: PASS — 14/21 profitable pairs.


## 6. Average winning-pair profit versus average losing-pair loss

Requirement: average profit of profitable pairs should be approximately equal to or greater than average absolute loss of losing pairs. No discretionary tolerance is applied.

In [7]:
comparison = pd.DataFrame({'average_winning_pair_profit_usd': [mean_profit], 'average_losing_pair_loss_abs_usd': [mean_loss], 'profit_to_loss_ratio': [mean_profit / mean_loss if mean_loss > 0 else np.nan], 'passed': [mean_comparison]})
show_table(comparison)

 average_winning_pair_profit_usd  average_losing_pair_loss_abs_usd  profit_to_loss_ratio  passed
                        298.7907                          124.1386                2.4069    True


## 7. Rule assessment conclusion

In [8]:
preliminary_pass = bool(pair_report['at_least_100'].all() and len(winners) >= 14 and mean_comparison)
print('Preliminary 21-pair screen:', 'PASS' if preliminary_pass else 'FAIL')
print(f"Full pair-level conditions (sample size, count balance and PF balance): {pair_report['full_pair_rules_ok'].sum()}/21 pass.")
print('Passing pairs:', ', '.join(pair_report.loc[pair_report['full_pair_rules_ok'], 'symbol'].str.replace('FX:', '', regex=False)))
print('The optimization-exclusion condition is omitted as instructed.')

Preliminary 21-pair screen: PASS
Full pair-level conditions (sample size, count balance and PF balance): 11/21 pass.
Passing pairs: AUDCHF, AUDUSD, EURAUD, EURCAD, GBPAUD, GBPCAD, GBPCHF, GBPNZD, NZDCAD, USDCAD, USDCHF
The optimization-exclusion condition is omitted as instructed.


## 8. Annual realized performance

Profit is assigned to the year of exit. Each pair uses its available history. CADCHF begins on 31 October 2008; the other pairs begin on 15 September 2008. All coverage ends on 20 November 2026. The first and last calendar years are incomplete.

The matrix contains net realized P&L in USD. A dash means no coverage, not zero profit. Combined totals sum independent backtests; they do not represent a capital-constrained shared account.

In [9]:
checks['coverage_start'] = pd.to_datetime(checks['coverage_start'])
checks['coverage_end'] = pd.to_datetime(checks['coverage_end'])

def performance_by_period(frequency):
    rows = []
    for _, coverage in checks.iterrows():
        symbol = coverage['symbol']
        data = closed[closed['symbol'] == symbol].copy()
        data['period'] = data['exit_time'].dt.to_period(frequency)
        for period in pd.period_range(coverage['coverage_start'], coverage['coverage_end'], freq=frequency):
            pnl = data.loc[data['period'] == period, 'pnl_usd']
            rows.append({
                'symbol': symbol, 'period': str(period),
                'partial_period': bool(coverage['coverage_start'] > period.start_time or coverage['coverage_end'] < period.end_time),
                'closed_trades': len(pnl), 'wins': int((pnl > 0).sum()), 'losses': int((pnl < 0).sum()),
                'breakeven': int((pnl == 0).sum()),
                'gross_profit_usd': pnl[pnl > 0].sum(), 'gross_loss_abs_usd': -pnl[pnl < 0].sum(),
                'net_pnl_usd': pnl.sum(), 'win_rate_pct': (pnl > 0).mean() * 100 if len(pnl) else np.nan,
                'profit_factor': profit_factor(pnl), 'average_trade_usd': pnl.mean() if len(pnl) else np.nan})
    return pd.DataFrame(rows)

yearly = performance_by_period('Y')
monthly = performance_by_period('M')

def combined_performance(report):
    result = report.groupby('period', as_index=False).agg(
        covered_pairs=('symbol', 'nunique'), partial_pairs=('partial_period', 'sum'),
        closed_trades=('closed_trades', 'sum'), wins=('wins', 'sum'), losses=('losses', 'sum'),
        gross_profit_usd=('gross_profit_usd', 'sum'), gross_loss_abs_usd=('gross_loss_abs_usd', 'sum'),
        net_pnl_usd=('net_pnl_usd', 'sum'))
    result['win_rate_pct'] = result['wins'].div(result['closed_trades'].replace(0, np.nan)) * 100
    result['profit_factor'] = result['gross_profit_usd'].div(result['gross_loss_abs_usd'].replace(0, np.nan))
    result.loc[(result['gross_loss_abs_usd'] == 0) & (result['gross_profit_usd'] > 0), 'profit_factor'] = np.inf
    return result

annual_total = combined_performance(yearly)
monthly_total = combined_performance(monthly)
for name, report in [('yearly_per_pair', yearly), ('monthly_per_pair', monthly), ('yearly_all_pairs', annual_total), ('monthly_all_pairs', monthly_total)]:
    report.to_csv(output_folder / (name + '.csv'), index=False, encoding='utf-8-sig')
for report in [yearly, monthly]:
    assert report['closed_trades'].sum() == len(closed)
    actual = report.groupby('symbol')['net_pnl_usd'].sum().sort_index()
    expected = closed.groupby('symbol')['pnl_usd'].sum().sort_index()
    assert np.allclose(actual, expected)

annual_matrix = yearly.pivot(index='period', columns='symbol', values='net_pnl_usd')
annual_matrix.columns = annual_matrix.columns.str.replace('FX:', '', regex=False)
print('Annual net realized P&L by pair (USD):')
print(annual_matrix.round(2).to_string(na_rep='—'))
print('\nCombined annual results:')
show_table(annual_total)
print(f"Annual reconciliation: PASS. Total realized P&L: ${closed['pnl_usd'].sum():,.2f}.")

Annual net realized P&L by pair (USD):
symbol  AUDCAD  AUDCHF  AUDNZD  AUDUSD  CADCHF  EURAUD  EURCAD  EURCHF  EURGBP  EURNZD  EURUSD  GBPAUD  GBPCAD  GBPCHF  GBPNZD  GBPUSD  NZDCAD  NZDCHF  NZDUSD  USDCAD  USDCHF
period                                                                                                                                                                        
2008     20.27  -42.25    7.42   56.24  -23.62  -18.61  -31.44    6.95  -49.38   15.56   32.76   60.29  126.37  -69.04   91.70 -108.27   54.06   60.74  -42.05  -28.97    5.06
2009     37.31  -56.38   43.32   27.05  -39.03  -63.65   26.69   87.00  -91.16   94.34  -46.09  178.21  -78.74 -148.16  -49.34 -184.83   29.14   59.56  -33.17  -56.13   63.64
2010     39.15   30.35   33.19   54.84   36.81  -35.11   -2.95    2.13  -20.76   64.52  -80.01   28.69   38.15   56.03   66.87  -12.08   47.60   20.19    4.85   72.93   30.22
2011     54.21  -95.43   44.17  -12.32 -167.34 -124.78   91.51  -66.81  -10.86  -90.42

## 9. Monthly realized performance

Each year is shown separately. The pair columns report monthly net realized P&L in USD; the final columns report total P&L, closed-trade count, win rate, PF and coverage. `partial_pairs` counts pairs without full calendar-month coverage. Months within the stated coverage with no closed trades have zero realized P&L; unavailable performance ratios remain blank.

These are realized results, not month-end equity returns. Price gaps, missing candles and floating P&L cannot be established from trade exports alone.

In [10]:
monthly_matrix = monthly.pivot(index='period', columns='symbol', values='net_pnl_usd')
monthly_matrix.columns = monthly_matrix.columns.str.replace('FX:', '', regex=False)
summary_columns = ['period', 'net_pnl_usd', 'closed_trades', 'win_rate_pct', 'profit_factor', 'covered_pairs', 'partial_pairs']
monthly_view = monthly_matrix.join(monthly_total[summary_columns].set_index('period').rename(columns={'net_pnl_usd': 'TOTAL_USD', 'profit_factor': 'PF'}))
for year in sorted({period[:4] for period in monthly_view.index}):
    print('\n' + year)
    print(monthly_view.loc[monthly_view.index.str.startswith(year)].round(2).to_string(na_rep='—'))
print('\nMonthly reconciliation: PASS. Every pair matches both its annual totals and closed-trade P&L.')


2008
         AUDCAD  AUDCHF  AUDNZD  AUDUSD  CADCHF  EURAUD  EURCAD  EURCHF  EURGBP  EURNZD  EURUSD  GBPAUD  GBPCAD  GBPCHF  GBPNZD  GBPUSD  NZDCAD  NZDCHF  NZDUSD  USDCAD  USDCHF  TOTAL_USD  closed_trades  win_rate_pct    PF  covered_pairs  partial_pairs
period                                                                                                                                                                                                                                                     
2008-09   -8.66  -16.62   -8.91    5.99       —  -29.07    3.52  -15.64    6.27   39.75  -26.34    8.72   19.56   26.28   29.76   -7.28   11.29    7.50  -10.37   12.51   -1.09      47.17             85         48.24  1.12             20             20
2008-10   15.07  -72.95    3.86  -36.04    0.00  -68.78  -12.36   28.05  -34.58  -32.67   71.03  -11.94   70.02  -83.95   53.15 -115.02   24.70  -10.73  -28.47  -50.48    9.70    -282.39            208         44.71  0.83             21  